In [ ]:
# Databricks notebook source
# MAGIC
# MAGIC **What we ingest:**
# MAGIC - Interest rates (Federal Funds Rate)
# MAGIC - Inflation (CPI)
# MAGIC - GDP Growth Rate
# MAGIC - Unemployment Rate
# MAGIC
# MAGIC **Design:**
# MAGIC - Secrets read from Azure Key Vault via Databricks Secret Scope
# MAGIC - Raw JSON landed as-is (append-only, no transformation)
# MAGIC - Partitioned by ingestion date
# MAGIC - Full audit columns on every record

In [ ]:
# Read all secrets from Key Vault via Databricks Secret Scope
FRED_API_KEY     = dbutils.secrets.get(scope="retail-banking-scope", key="fred-api-key")
SP_CLIENT_ID     = dbutils.secrets.get(scope="retail-banking-scope", key="sp-client-id")
SP_TENANT_ID     = dbutils.secrets.get(scope="retail-banking-scope", key="sp-tenant-id")
SP_CLIENT_SECRET = dbutils.secrets.get(scope="retail-banking-scope", key="sp-client-secret")

# ADLS Gen2 paths
STORAGE_ACCOUNT  = "retailbankingdl"
CONTAINER_BRONZE = "bronze"
ADLS_BRONZE_PATH = f"abfss://{CONTAINER_BRONZE}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# FRED API
FRED_BASE_URL = "https://api.stlouisfed.org/fred/series/observations"

print("✅ Secrets loaded successfully — no values printed for security")

In [ ]:
spark.conf.set(
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    SP_CLIENT_ID
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    SP_CLIENT_SECRET
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{SP_TENANT_ID}/oauth2/token"
)

print("✅ ADLS Gen2 connection configured via Service Principal")

In [ ]:
import requests
import json
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType


def fetch_fred_series(series_id: str, series_name: str) -> dict:
    """
    Fetch a FRED economic data series.
    Returns last 5 years of observations.
    """
    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "observation_start": "2019-01-01",
        "sort_order": "desc"
    }

    response = requests.get(FRED_BASE_URL, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    if "error_code" in data:
        raise ValueError(f"FRED API error for {series_id}: {data.get('error_message', 'Unknown error')}")

    obs_count = data.get("count", 0)
    print(f"  📊 {series_name} ({series_id}): {obs_count} observations fetched")

    return data


def land_to_bronze(data: dict, source_name: str, table_name: str):
    """
    Land raw JSON as a single record into Bronze Delta table.
    - Append-only (never overwrite raw data)
    - Partitioned by ingestion_date
    - Full audit columns
    """
    ingestion_ts   = datetime.utcnow().isoformat()
    ingestion_date = datetime.utcnow().strftime("%Y-%m-%d")

    schema = StructType([
        StructField("raw_json",       StringType(), False),
        StructField("source",         StringType(), False),
        StructField("ingestion_ts",   StringType(), False),
        StructField("ingestion_date", StringType(), False),
    ])

    row = [(
        json.dumps(data),
        source_name,
        ingestion_ts,
        ingestion_date
    )]

    df = spark.createDataFrame(row, schema)

    output_path = f"{ADLS_BRONZE_PATH}/{table_name}"

    (df.write
       .format("delta")
       .mode("append")
       .partitionBy("ingestion_date")
       .save(output_path))

    print(f"  ✅ Landed → {output_path} (partition: {ingestion_date})")
    return output_path

In [ ]:
print("Fetching Federal Funds Rate...")

fedfunds_data = fetch_fred_series(
    series_id="FEDFUNDS",
    series_name="Federal Funds Rate"
)

land_to_bronze(
    data=fedfunds_data,
    source_name="fred_interest_rate_fedfunds",
    table_name="fred_interest_rate"
)

In [ ]:
print("Fetching CPI Inflation data...")

cpi_data = fetch_fred_series(
    series_id="CPIAUCSL",
    series_name="Consumer Price Index (CPI)"
)

land_to_bronze(
    data=cpi_data,
    source_name="fred_inflation_cpi",
    table_name="fred_inflation"
)

In [ ]:
print("Fetching GDP Growth Rate...")

gdp_data = fetch_fred_series(
    series_id="A191RL1Q225SBEA",
    series_name="GDP Growth Rate"
)

land_to_bronze(
    data=gdp_data,
    source_name="fred_gdp_growth",
    table_name="fred_gdp"
)

In [ ]:
print("Fetching Unemployment Rate...")

unemployment_data = fetch_fred_series(
    series_id="UNRATE",
    series_name="Unemployment Rate"
)

land_to_bronze(
    data=unemployment_data,
    source_name="fred_unemployment",
    table_name="fred_unemployment"
)

In [ ]:
print("\n--- FRED Bronze Layer Contents ---")

fred_tables = [
    "fred_interest_rate",
    "fred_inflation",
    "fred_gdp",
    "fred_unemployment"
]

for table in fred_tables:
    path = f"{ADLS_BRONZE_PATH}/{table}"
    df = spark.read.format("delta").load(path)
    count = df.count()
    print(f"\n📁 {table}: {count} record(s)")
    df.select("source", "ingestion_ts", "ingestion_date").show(truncate=False)

In [ ]:
for table in fred_tables:
    path = f"{ADLS_BRONZE_PATH}/{table}"
    print(f"\n📋 History for {table}:")
    spark.sql(f"DESCRIBE HISTORY delta.`{path}`").select(
        "version", "timestamp", "operation"
    ).show(truncate=False)

In [ ]:
print("=" * 50)
print("FRED Bronze Ingestion — COMPLETE")
print("=" * 50)
print("  ✅ Federal Funds Interest Rate")
print("  ✅ CPI Inflation")
print("  ✅ GDP Growth Rate")
print("  ✅ Unemployment Rate")
print("=" * 50)
print(f"All tables landed in: {ADLS_BRONZE_PATH}")
print("=" * 50)